# 🏪 Multi-Branch Sari-Sari Store Management System
### Aling Puring Store — Quezon City
**Course:** IT 105 / CC 103 | **Problem Set #1**

---
**Features:**
- ✅ Manage multiple branches with separate inventories
- ✅ Record daily sales transactions
- ✅ Automatically update stock levels
- ✅ Generate sales reports (per branch and overall)
- ✅ Save and load data using file handling (JSON)

---
> **How to use:** Run each cell from top to bottom. The last cell launches the main menu.

## Cell 1 — Imports & File Handling

In [1]:
import json
import os
from datetime import datetime

DATA_FILE = "store_data.json"

DEFAULT_DATA = {
    "branches": {
        "Branch A - Tandang Sora": {
            "inventory": {
                "Tide Powder (small)": {"price": 7.00,  "stock": 50},
                "Bear Brand (sachet)": {"price": 9.00,  "stock": 40},
                "Lucky Me Noodles":    {"price": 10.00, "stock": 60},
                "Coke 8oz":            {"price": 12.00, "stock": 30},
                "Chippy":              {"price": 8.00,  "stock": 45},
                "Eggs (per piece)":    {"price": 8.00,  "stock": 100},
                "Rice (per kilo)":     {"price": 52.00, "stock": 20},
                "Alaska Milk (small)": {"price": 13.00, "stock": 35}
            },
            "transactions": []
        },
        "Branch B - Batasan Hills": {
            "inventory": {
                "Tide Powder (small)": {"price": 7.00,  "stock": 30},
                "Bear Brand (sachet)": {"price": 9.00,  "stock": 25},
                "Lucky Me Noodles":    {"price": 10.00, "stock": 50},
                "Coke 8oz":            {"price": 12.00, "stock": 20},
                "Chippy":              {"price": 8.00,  "stock": 40},
                "Eggs (per piece)":    {"price": 8.00,  "stock": 80},
                "Rice (per kilo)":     {"price": 52.00, "stock": 15},
                "Alaska Milk (small)": {"price": 13.00, "stock": 20}
            },
            "transactions": []
        },
        "Branch C - Commonwealth": {
            "inventory": {
                "Tide Powder (small)": {"price": 7.00,  "stock": 35},
                "Bear Brand (sachet)": {"price": 9.00,  "stock": 30},
                "Lucky Me Noodles":    {"price": 10.00, "stock": 45},
                "Coke 8oz":            {"price": 12.00, "stock": 25},
                "Chippy":              {"price": 8.00,  "stock": 50},
                "Eggs (per piece)":    {"price": 8.00,  "stock": 90},
                "Rice (per kilo)":     {"price": 52.00, "stock": 18},
                "Alaska Milk (small)": {"price": 13.00, "stock": 28}
            },
            "transactions": []
        }
    }
}

def save_data(data):
    """Save store data to a JSON file."""
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=4)
    print("\n  Data saved successfully.\n")

def load_data():
    """Load store data from JSON file, or use defaults if none found."""
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, "r") as f:
            print("  Loaded existing data from file.\n")
            return json.load(f)
    else:
        print("  No save file found. Starting with default data.\n")
        return DEFAULT_DATA.copy()

print("Cell 1 loaded: Imports & File Handling")

Cell 1 loaded: Imports & File Handling


## Cell 2 — Display Helpers

In [2]:
def print_header(title):
    width = 58
    print("\n" + "="*width)
    print(f"  {title}")
    print("="*width)

def print_branch_menu(data):
    print_header("SELECT A BRANCH")
    branches = list(data["branches"].keys())
    for i, name in enumerate(branches, 1):
        print(f"  [{i}] {name}")
    print("  [0] Back")
    return branches

print("Cell 2 loaded: Display Helpers")

Cell 2 loaded: Display Helpers


## Cell 3 — Inventory Management

In [3]:
def view_inventory(data, branch_name):
    inventory = data["branches"][branch_name]["inventory"]
    print_header(f"INVENTORY -- {branch_name}")
    print(f"  {'#':<4} {'Product':<25} {'Price':>8}  {'Stock':>6}")
    print("  " + "-"*50)
    for i, (product, info) in enumerate(inventory.items(), 1):
        stock_label = str(info['stock'])
        if info["stock"] <= 10:
            stock_label += "  [LOW STOCK]"
        print(f"  {i:<4} {product:<25} P{info['price']:>7.2f}  {stock_label}")
    print()

def add_product(data, branch_name):
    print_header(f"ADD PRODUCT -- {branch_name}")
    name = input("  Product name: ").strip()
    if not name:
        print("  ERROR: Product name cannot be empty.")
        return
    if name in data["branches"][branch_name]["inventory"]:
        print(f"  ERROR: '{name}' already exists. Use Restock instead.")
        return
    try:
        price = float(input("  Price (P): "))
        stock = int(input("  Initial stock (qty): "))
        if price <= 0 or stock < 0:
            raise ValueError
    except ValueError:
        print("  ERROR: Invalid price or stock value.")
        return
    data["branches"][branch_name]["inventory"][name] = {"price": price, "stock": stock}
    print(f"\n  SUCCESS: '{name}' added to {branch_name}.")
    save_data(data)

def restock_product(data, branch_name):
    inventory = data["branches"][branch_name]["inventory"]
    view_inventory(data, branch_name)
    products = list(inventory.keys())
    try:
        choice = int(input("  Enter product number to restock: ")) - 1
        if choice < 0 or choice >= len(products):
            print("  ERROR: Invalid choice.")
            return
        qty = int(input("  Quantity to add: "))
        if qty <= 0:
            raise ValueError
    except ValueError:
        print("  ERROR: Invalid input.")
        return
    product = products[choice]
    inventory[product]["stock"] += qty
    print(f"\n  SUCCESS: Restocked '{product}' (+{qty}). New stock: {inventory[product]['stock']}")
    save_data(data)

print("Cell 3 loaded: Inventory Management")

Cell 3 loaded: Inventory Management


## Cell 4 — Sales Transactions

In [4]:
def record_sale(data, branch_name):
    inventory = data["branches"][branch_name]["inventory"]
    view_inventory(data, branch_name)
    products = list(inventory.keys())
    cart = []
    print("  Enter items sold (type 0 when done):\n")

    while True:
        try:
            choice = int(input("  Product number (0 to finish): "))
            if choice == 0:
                break
            if choice < 1 or choice > len(products):
                print("  ERROR: Invalid number.")
                continue
            product = products[choice - 1]
            qty = int(input(f"  Quantity of '{product}': "))
            if qty <= 0:
                print("  ERROR: Quantity must be positive.")
                continue
            if qty > inventory[product]["stock"]:
                print(f"  ERROR: Not enough stock! Available: {inventory[product]['stock']}")
                continue
            cart.append({"product": product, "qty": qty, "price": inventory[product]["price"]})
            print(f"  Added {qty}x {product} to cart.\n")
        except ValueError:
            print("  ERROR: Please enter a valid number.")

    if not cart:
        print("  No items recorded.")
        return

    total = 0
    print_header("RECEIPT")
    print(f"  Branch : {branch_name}")
    print(f"  Date   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("  " + "-"*40)
    for item in cart:
        subtotal = item["qty"] * item["price"]
        total += subtotal
        inventory[item["product"]]["stock"] -= item["qty"]
        print(f"  {item['product']:<25} x{item['qty']}  P{subtotal:.2f}")
    print("  " + "-"*40)
    print(f"  {'TOTAL':>30}  P{total:.2f}\n")

    transaction = {
        "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "items": cart,
        "total": total
    }
    data["branches"][branch_name]["transactions"].append(transaction)
    save_data(data)

print("Cell 4 loaded: Sales Transactions")

Cell 4 loaded: Sales Transactions


## Cell 5 — Reports

In [5]:
def branch_report(data, branch_name):
    branch = data["branches"][branch_name]
    transactions = branch["transactions"]
    print_header(f"SALES REPORT -- {branch_name}")
    if not transactions:
        print("  No transactions recorded yet.\n")
        return
    total_revenue = 0
    product_sales = {}
    for txn in transactions:
        total_revenue += txn["total"]
        for item in txn["items"]:
            p = item["product"]
            product_sales[p] = product_sales.get(p, 0) + item["qty"]
    print(f"  Total Transactions : {len(transactions)}")
    print(f"  Total Revenue      : P{total_revenue:.2f}")
    print(f"\n  {'Product':<25} {'Qty Sold':>10}")
    print("  " + "-"*37)
    sorted_products = sorted(product_sales.items(), key=lambda x: x[1], reverse=True)
    for product, qty in sorted_products:
        print(f"  {product:<25} {qty:>10}")
    if sorted_products:
        print(f"\n  Fast-moving : {sorted_products[0][0]}")
        print(f"  Slow-moving : {sorted_products[-1][0]}")
    print()

def overall_report(data):
    print_header("OVERALL STORE REPORT -- ALL BRANCHES")
    grand_total = 0
    all_product_sales = {}
    for branch_name, branch in data["branches"].items():
        branch_total = sum(t["total"] for t in branch["transactions"])
        grand_total += branch_total
        txn_count = len(branch["transactions"])
        print(f"  {branch_name}")
        print(f"    Transactions: {txn_count}  |  Revenue: P{branch_total:.2f}")
        for txn in branch["transactions"]:
            for item in txn["items"]:
                p = item["product"]
                all_product_sales[p] = all_product_sales.get(p, 0) + item["qty"]
    print("\n  " + "-"*40)
    print(f"  GRAND TOTAL REVENUE : P{grand_total:.2f}")
    if all_product_sales:
        sorted_all = sorted(all_product_sales.items(), key=lambda x: x[1], reverse=True)
        print(f"\n  Top Seller (all branches): {sorted_all[0][0]} ({sorted_all[0][1]} pcs)")
        print(f"  Least Sold (all branches): {sorted_all[-1][0]} ({sorted_all[-1][1]} pcs)")
    print()

def low_stock_report(data):
    print_header("LOW STOCK ALERT (10 units or less)")
    found = False
    for branch_name, branch in data["branches"].items():
        for product, info in branch["inventory"].items():
            if info["stock"] <= 10:
                print(f"  [!] {branch_name} | {product} -- {info['stock']} left")
                found = True
    if not found:
        print("  All branches have sufficient stock.")
    print()

print("Cell 5 loaded: Reports")

Cell 5 loaded: Reports


## Cell 6 — Menu System

In [6]:
def branch_menu(data, branch_name):
    while True:
        print_header(f"BRANCH MENU -- {branch_name}")
        print("  [1] View Inventory")
        print("  [2] Add New Product")
        print("  [3] Restock Product")
        print("  [4] Record Sale Transaction")
        print("  [5] View Branch Sales Report")
        print("  [0] Back to Main Menu")
        choice = input("\n  Choose: ").strip()
        if choice == "1":
            view_inventory(data, branch_name)
        elif choice == "2":
            add_product(data, branch_name)
        elif choice == "3":
            restock_product(data, branch_name)
        elif choice == "4":
            record_sale(data, branch_name)
        elif choice == "5":
            branch_report(data, branch_name)
        elif choice == "0":
            break
        else:
            print("  ERROR: Invalid choice.\n")

def reports_menu(data):
    while True:
        print_header("REPORTS MENU")
        print("  [1] Overall Store Report (All Branches)")
        print("  [2] Per-Branch Sales Report")
        print("  [3] Low Stock Alert")
        print("  [0] Back to Main Menu")
        choice = input("\n  Choose: ").strip()
        if choice == "1":
            overall_report(data)
        elif choice == "2":
            branches = print_branch_menu(data)
            bc = input("\n  Choose branch: ").strip()
            if bc.isdigit():
                idx = int(bc) - 1
                if 0 <= idx < len(branches):
                    branch_report(data, branches[idx])
        elif choice == "3":
            low_stock_report(data)
        elif choice == "0":
            break
        else:
            print("  ERROR: Invalid choice.\n")

def main_menu(data):
    while True:
        print_header("ALING PURING'S SARI-SARI STORE SYSTEM")
        print("  Multi-Branch Management System")
        print()
        print("  [1] Manage a Branch")
        print("  [2] Reports")
        print("  [3] Save Data")
        print("  [0] Exit")
        choice = input("\n  Choose: ").strip()
        if choice == "1":
            branches = print_branch_menu(data)
            bc = input("\n  Choose branch: ").strip()
            if bc.isdigit():
                idx = int(bc) - 1
                if 0 <= idx < len(branches):
                    branch_menu(data, branches[idx])
                elif int(bc) == 0:
                    pass
                else:
                    print("  ERROR: Invalid choice.\n")
        elif choice == "2":
            reports_menu(data)
        elif choice == "3":
            save_data(data)
        elif choice == "0":
            save_data(data)
            print("  Salamat! Goodbye from Aling Puring's Store!\n")
            break
        else:
            print("  ERROR: Invalid choice.\n")

print("Cell 6 loaded: Menu System")

Cell 6 loaded: Menu System


## Cell 7 — Run the Program

> **Run this cell last** to launch the system. All cells above must be run first (or use **Kernel > Restart & Run All**).

In [ ]:
print("="*58)
print("   ALING PURING'S SARI-SARI STORE SYSTEM")
print("   Multi-Branch | Quezon City")
print("="*58 + "\n")

store_data = load_data()
main_menu(store_data)

   ALING PURING'S SARI-SARI STORE SYSTEM
   Multi-Branch | Quezon City

  No save file found. Starting with default data.


  ALING PURING'S SARI-SARI STORE SYSTEM
  Multi-Branch Management System

  [1] Manage a Branch
  [2] Reports
  [3] Save Data
  [0] Exit

  SELECT A BRANCH
  [1] Branch A - Tandang Sora
  [2] Branch B - Batasan Hills
  [3] Branch C - Commonwealth
  [0] Back
